# 01. Pain Narratives Mapping, Description & Author Demographics

This notebook establishes traceability for pain narratives across different data sources:

**Data Sources:**
- **Excel File**: Original validated narratives with author demographics
- **Batch Processing** (groups 35-36): LLM evaluations of narratives
- **Expert UI** (groups 16-32): Human expert evaluations via web interface

**Challenge**: The same narrative text receives different `narrative_id` values in each experiment group.

**Solution**: Use `narrative_hash` (SHA-256) and `word_count` to map narratives across sources.

**Output**: Mapping table stored in database schema `pain_narratives_acm_202512` for all future analyses.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlmodel import Session, select, text
from pain_narratives.core.database import DatabaseManager
from pain_narratives.db.models_sqlmodel import Narrative, ExperimentList, EvaluationResult, User

# Configuration
BATCH_GROUPS = [35, 36]  # LLM batch processing
EXPERT_GROUP = [16, 32]    # Expert UI (>= 16 and <= 32)
# EXPERT_GROUP_MAX = 32

NEW_SCHEMA = 'pain_narratives_acm_202512'

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
EXCEL_FILE = DATA_DIR / 'pain_narratives_20251126.xlsx'

print(f'Excel file exists: {EXCEL_FILE.exists()}')
print(f'Data directory: {DATA_DIR}')

## 1. Database Setup

Create new schema for clean, validated data

In [ ]:
db_manager = DatabaseManager()
engine = db_manager.engine

# Drop the schema if it exists
with Session(engine) as session:
    session.exec(text(f'DROP SCHEMA IF EXISTS {NEW_SCHEMA} CASCADE'))
    session.commit()

# Create schema if it doesn't exist
with Session(engine) as session:
    session.exec(text(f'CREATE SCHEMA IF NOT EXISTS {NEW_SCHEMA}'))
    session.commit()
    print(f'✓ Schema {NEW_SCHEMA} ready')

## 2. Load and Validate Excel Data

Load narratives from Excel file, filtering only valid entries (valid = 1)

In [ ]:
# Load Excel file
df_excel_raw = pd.read_excel(EXCEL_FILE)

print(f'Total rows in Excel: {len(df_excel_raw)}')
print(f'Total columns: {len(df_excel_raw.columns)}')
print()

# Check for 'valid' column
if 'valid' in df_excel_raw.columns:
    print(f'Valid column found:')
    print(df_excel_raw['valid'].value_counts())
    df_excel = df_excel_raw[df_excel_raw['valid'] == 1].copy()
    print(f'\nFiltered to valid=1: {len(df_excel)} rows')
else:
    print('⚠️  No "valid" column found - using all rows')
    df_excel = df_excel_raw.copy()

### 2.1 Column Name Mapping

Map Spanish questionnaire column names to standardized snake_case format

In [ ]:
import re

# ============================================================================
# COLUMN NAME MAPPING: Spanish (Excel) -> snake_case (Database)
# ============================================================================

# Core identification
CORE_COLUMNS = {
    'valid': 'is_valid',
    'ID': 'participant_id',
    'Start Date': 'survey_start_date',
    'End Date': 'survey_end_date',
    'Response Type': 'response_type',
    'Progress': 'survey_progress',
    'Duration (in seconds)': 'survey_duration_seconds',
    'Finished': 'survey_finished',
    'Recorded Date': 'survey_recorded_date',
    'Response ID': 'qualtrics_response_id',
    'Distribution Channel': 'distribution_channel',
    'User Language': 'user_language',
}

# Consent and screening
CONSENT_COLUMNS = {
    'He sido informado/a sobre el proyecto, su finalidad y sobre los datos que se recogerán, y he consentido la participación en este estudio.': 'consent_informed',
    'Acepto el tratamiento de los datos personales en el marco del proyecto, y con ello inicio la encuesta:': 'consent_data_treatment',
    '¿Ha tenido dolor de forma constante o con episodios significativos en los tres últimos meses?': 'chronic_pain_3months',
    'Si no ha tenido dolor durante los últimos tres meses de forma constante o con episodios significativos, le agradecemos su colaboración, pero en este estudio estamos interesados en la experiencia de personas con dolor crónico. Confirme su selección:': 'chronic_pain_confirmation',
}

# Pain history
PAIN_HISTORY_COLUMNS = {
    '¿Cuántos años hace que sufre de dolor?': 'years_with_pain',
    '¿Cuántos años hace que le diagnosticaron?': 'years_since_diagnosis',
    '¿Cuál es la causa principal de su dolor?\nPor favor, señale la principal causa de su dolor. - Selected Choice': 'pain_cause_primary',
    '¿Cuál es la causa principal de su dolor?\nPor favor, señale la principal causa de su dolor. - Otro - Texto': 'pain_cause_other_text',
    'Por favor, señala las zona en la que se localiza el dolor crónico (marca todas las que sean aplica': 'pain_location_zones',
}

# Demographics
DEMOGRAPHICS_COLUMNS = {
    'Género': 'gender',
    'Edad': 'age',
    'Estado civil': 'marital_status',
    'Nivel de estudios': 'education_level',
    'País de residencia actual': 'country_residence',
    'País de nacimiento': 'country_birth',
    'Situación laboral (principal) - Selected Choice': 'employment_status',
    'Situación laboral (principal) - Otra Situación (indicar) - Texto': 'employment_other_text',
}

# Narrative
NARRATIVE_COLUMNS = {
    'narrative': 'narrative_text',
}

# BPI (Brief Pain Inventory) - Using original BPI item codes
# Interference Scale: Q1 items 1,2,3,5,6,7 (0-10 scale)
# Pain Intensity: Q2,Q3,Q4,Q5 items 8,9,10,11 (0-10 scale)
BPI_INTERFERENCE_COLUMNS = {
    '. - 1. Actividad general': 'bpiq11',  # Q1 Item 1
    '. - 2. Estado de ánimo': 'bpiq12',  # Q1 Item 2
    '. - 3. Capacidad de andar': 'bpiq13',  # Q1 Item 3
    '. - 4. Trabajo normal (incluye tanto el trabajo fuera de casa como el doméstico)': 'bpiq14',  # Q1 Item 4
    '. - 5. Relaciones con otras personas': 'bpiq15',  # Q1 Item 5
    '. - 6. Sueño': 'bpiq16',  # Q1 Item 6
    '. - 7. Disfrute de la vida': 'bpiq17',  # Q1 Item 7
}

BPI_INTENSITY_COLUMNS = {
    'Puntúe EL PEOR DOLOR QUE HA SENTIDO EN LAS ÚLTIMAS 24 HORAS del 0 al 10, donde 0 representa "sin dolor" y 10 representa "el peor dolor que se pueda imaginar". - 7': 'bpiq28',  # Q2 Item 8
    'El DOLOR MÁS LEVE QUE HA SENTIDO EN LAS ÚLTIMAS 24 HORAS del 0 al 10, donde 0 representa "sin dolor" y 10 representa "el peor dolor que se pueda imaginar". - 7': 'bpiq39',  # Q3 Item 9
    'El DOLOR PROMEDIO EN LAS ÚLTIMAS 24 HORAS del 0 al 10, donde 0 representa "sin dolor" y 10 representa "el peor dolor que se pueda imaginar". - 7': 'bpiq410',  # Q4 Item 10
    'Puntúe EL DOLOR AHORA MISMO del 0 al 10, donde 0 representa "sin dolor" y 10 representa "el peor dolor que se pueda imaginar". - 7': 'bpiq511',  # Q5 Item 11
}

# PCS (Pain Catastrophizing Scale) - 13 items (0-4 scale)
# Using format: pcs_01, pcs_02, etc.
PCS_COLUMNS = {
    'Cuando tienes dolor... - 1. Cuando tienes dolor, ¿estás todo el rato preocupado y pensando en si el dolor desaparecerá?': 'pcs_01',
    'Cuando tienes dolor... - 2. Cuando tienes dolor, ¿sientes que no puedes continuar así?': 'pcs_02',
    'Cuando tienes dolor... - 3. Cuando tienes dolor, ¿te encuentras fatal y piensas que nunca mejorará?': 'pcs_03',
    'Cuando tienes dolor... - 4. Cuando tienes dolor, ¿es espantoso y sientes como si el dolor te controlase?': 'pcs_04',
    'Cuando tienes dolor... - 5. Cuando tienes dolor, ¿sientes que no lo puedes soportar más?': 'pcs_05',
    'Cuando tienes dolor... - 6. Cuando tienes dolor, ¿tienes miedo de que el dolor empeore?': 'pcs_06',
    'Cuando tienes dolor... - 7. Cuando tienes dolor, ¿piensas continuamente en otras veces en las que has tenido dolor?': 'pcs_07',
    'Cuando tienes dolor... - 8. Cuando tienes dolor, ¿quieres desesperadamente que el dolor desaparezca?': 'pcs_08',
    'Cuando tienes dolor... - 9. Cuando tienes dolor, ¿no puedes dejar de pensar en él?': 'pcs_09',
    'Cuando tienes dolor... - 10. Cuando tienes dolor, ¿piensas continuamente cuanto te duele?': 'pcs_10',
    'Cuando tienes dolor... - 11. Cuando tienes dolor, ¿piensas continuamente en lo mucho que deseas que el dolor se acabe?': 'pcs_11',
    'Cuando tienes dolor... - 12. Cuando tienes dolor, ¿no hay nada que tu puedas hacer para que el dolor desaparezca, o al menos disminuya?': 'pcs_12',
    'Cuando tienes dolor... - 13. Cuando tienes dolor, ¿te preguntas si puede pasar alguna cosa grave?': 'pcs_13',
}

# TSK-11SV (Tampa Scale of Kinesiophobia) - 11 items (1-4 scale)
# Using format: tsk_01, tsk_02, etc.
TSK_COLUMNS = {
    '. - 1. Tengo miedo de lesionarme si hago ejercicio físico': 'tsk_01',
    '. - 2.\tSi me dejara vencer por el dolor, el dolor aumentaría': 'tsk_02',
    '. - 3.\tMi cuerpo me está diciendo que tengo algo serio': 'tsk_03',
    '. - 4. Tener dolor siempre quiere decir que en el cuerpo hay una lesión': 'tsk_04',
    '. - 5.\tTengo miedo a lesionarme sin querer': 'tsk_05',
    '. - 6. Lo más seguro para evitar que aumente el dolor es tener cuidado y no hacer movimientos innecesarios': 'tsk_06',
    '. - 7. No me dolería tanto si no tuviese algo serio en mi cuerpo': 'tsk_07',
    '. - 8. El dolor me dice cuándo debo parar la actividad para no lesionarme': 'tsk_08',
    '. - 9. No es seguro para una persona con mi enfermedad hacer actividades físicas': 'tsk_09',
    '. - 10. No puedo hacer todo lo que la gente normal hace porque me podría lesionar con facilidad': 'tsk_10',
    '. - 11. Nadie debería hacer actividades físicas cuando tiene dolor': 'tsk_11',
}

# "Who am I" identity items (1-20)
IDENTITY_COLUMNS = {f'¿Quién soy yo? - {i}': f'identity_item_{str(i).zfill(2)}' for i in range(1, 21)}

# Future participation
PARTICIPATION_COLUMNS = {
    '¿Estaría interesado(a) en participar en futuros estudios sobre dolor?': 'interested_future_studies',
}

# Combine all mappings
ALL_COLUMN_MAPPINGS = {
    **CORE_COLUMNS,
    **CONSENT_COLUMNS,
    **PAIN_HISTORY_COLUMNS,
    **DEMOGRAPHICS_COLUMNS,
    **NARRATIVE_COLUMNS,
    **BPI_INTERFERENCE_COLUMNS,
    **BPI_INTENSITY_COLUMNS,
    **PCS_COLUMNS,
    **TSK_COLUMNS,
    **IDENTITY_COLUMNS,
    **PARTICIPATION_COLUMNS,
}

# Define questionnaire item lists for easy reference
BPI_INTERFERENCE_ITEMS = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq14', 'bpiq15', 'bpiq16', 'bpiq17']
BPI_INTENSITY_ITEMS = ['bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
PCS_ITEMS = [f'pcs_{str(i).zfill(2)}' for i in range(1, 14)]
TSK_ITEMS = [f'tsk_{str(i).zfill(2)}' for i in range(1, 12)]

print('✓ Column mappings defined')
print(f'  Core columns: {len(CORE_COLUMNS)}')
print(f'  Consent/Screening: {len(CONSENT_COLUMNS)}')
print(f'  Pain history: {len(PAIN_HISTORY_COLUMNS)}')
print(f'  Demographics: {len(DEMOGRAPHICS_COLUMNS)}')
print(f'  BPI-Interference: {len(BPI_INTERFERENCE_COLUMNS)} items → {BPI_INTERFERENCE_ITEMS}')
print(f'  BPI-Intensity: {len(BPI_INTENSITY_COLUMNS)} items → {BPI_INTENSITY_ITEMS}')
print(f'  PCS: {len(PCS_COLUMNS)} items → pcs_01 to pcs_13')
print(f'  TSK: {len(TSK_COLUMNS)} items → tsk_01 to tsk_11')
print(f'  Identity: {len(IDENTITY_COLUMNS)}')
print(f'  Total mapped columns: {len(ALL_COLUMN_MAPPINGS)}')
print(f'  Total Excel columns: {len(df_excel.columns)}')

### 2.2 Apply Column Mapping and Validate

In [ ]:
# Check which Excel columns exist in our mapping
existing_mapped_cols = {k: v for k, v in ALL_COLUMN_MAPPINGS.items() if k in df_excel.columns}
missing_in_excel = [k for k in ALL_COLUMN_MAPPINGS.keys() if k not in df_excel.columns]
unmapped_in_excel = [col for col in df_excel.columns if col not in ALL_COLUMN_MAPPINGS]

print('Column Mapping Status:')
print(f'  Excel columns mapped: {len(existing_mapped_cols)} / {len(ALL_COLUMN_MAPPINGS)}')
print(f'  Excel columns unmapped: {len(unmapped_in_excel)}')
print(f'  Expected but missing: {len(missing_in_excel)}')

if unmapped_in_excel:
    print(f'\nUnmapped Excel columns (first 5):')
    for col in unmapped_in_excel[:5]:
        print(f'  - {col}')

# Rename columns
df_excel_valid = df_excel.rename(columns=existing_mapped_cols)

print(f'\n✓ Renamed {len(existing_mapped_cols)} columns to snake_case format')
print(f'\nColumns available after mapping:')
print(f'  Total: {len(df_excel_valid.columns)}')

# Show sample of renamed columns
if 'participant_id' in df_excel_valid.columns:
    sample_cols = ['participant_id']
    if 'age' in df_excel_valid.columns:
        sample_cols.append('age')
    if 'gender' in df_excel_valid.columns:
        sample_cols.append('gender')
    if 'narrative_text' in df_excel_valid.columns:
        sample_cols.append('narrative_text')
    print(f'\nSample data ({", ".join(sample_cols)}):')
    print(df_excel_valid[sample_cols].head())

### 2.3 Process Questionnaire Data

Extract and validate questionnaire responses (PCS, BPI, TSK)

In [ ]:
# ============================================================================
# PCS (Pain Catastrophizing Scale) Processing
# Scale: 0=Nada, 1=Algo, 2=Bastante, 3=Mucho, 4=Muchísimo
# ============================================================================
print('='*80)
print('PCS (Pain Catastrophizing Scale) - Processing')
print('='*80)

# PCS response mapping
PCS_RESPONSE_MAPPING = {
    '0 Nada': 0, '1 Algo': 1, '2 Bastante': 2, '3 Mucho': 3, '4 Muchísimo': 4,
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4,
}

def convert_pcs_response(value):
    """Convert PCS text response to numeric (0-4)"""
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    
    # Try direct mapping
    if value_str in PCS_RESPONSE_MAPPING:
        return PCS_RESPONSE_MAPPING[value_str]
    
    # Extract leading digit
    match = re.match(r'^(\d+)', value_str)
    if match:
        num = int(match.group(1))
        if 0 <= num <= 4:
            return num
    return np.nan

# Get PCS item columns (pcs_01 to pcs_13)
pcs_item_cols = [col for col in df_excel_valid.columns if col in PCS_ITEMS]
print(f'Found {len(pcs_item_cols)} PCS items: {pcs_item_cols}')

# Convert PCS responses
for col in pcs_item_cols:
    df_excel_valid[col] = df_excel_valid[col].apply(convert_pcs_response)

# Calculate PCS total and subscales
if len(pcs_item_cols) >= 13:
    df_excel_valid['pcs_total'] = df_excel_valid[pcs_item_cols].sum(axis=1, min_count=13)
    
    # Subscales based on item numbers
    # Rumination: items 8, 9, 10, 11
    rumination_items = ['pcs_08', 'pcs_09', 'pcs_10', 'pcs_11']
    rumination_items = [col for col in rumination_items if col in df_excel_valid.columns]
    
    # Magnification: items 6, 7, 13
    magnification_items = ['pcs_06', 'pcs_07', 'pcs_13']
    magnification_items = [col for col in magnification_items if col in df_excel_valid.columns]
    
    # Helplessness: items 1, 2, 3, 4, 5, 12
    helplessness_items = ['pcs_01', 'pcs_02', 'pcs_03', 'pcs_04', 'pcs_05', 'pcs_12']
    helplessness_items = [col for col in helplessness_items if col in df_excel_valid.columns]
    
    if len(rumination_items) == 4:
        df_excel_valid['pcs_rumination'] = df_excel_valid[rumination_items].sum(axis=1, min_count=4)
    if len(magnification_items) == 3:
        df_excel_valid['pcs_magnification'] = df_excel_valid[magnification_items].sum(axis=1, min_count=3)
    if len(helplessness_items) == 6:
        df_excel_valid['pcs_helplessness'] = df_excel_valid[helplessness_items].sum(axis=1, min_count=6)
    
    print(f'\n✓ PCS scores computed')
    print(f'  Total score: Mean={df_excel_valid["pcs_total"].mean():.2f}, SD={df_excel_valid["pcs_total"].std():.2f}')
    print(f'  Range: [{df_excel_valid["pcs_total"].min():.0f}, {df_excel_valid["pcs_total"].max():.0f}]')
    print(f'  Valid responses: {df_excel_valid["pcs_total"].notna().sum()} / {len(df_excel_valid)}')
else:
    print(f'\n⚠️  Insufficient PCS items found ({len(pcs_item_cols)}/13)')

In [ ]:
# ============================================================================
# BPI (Brief Pain Inventory) Processing
# Scale: 0-10 (already numeric)
# ============================================================================
print('\n' + '='*80)
print('BPI (Brief Pain Inventory) - Processing')
print('='*80)

def convert_bpi_response(value):
    """Convert BPI response to numeric (0-10)"""
    if pd.isna(value):
        return np.nan
    try:
        num = float(value)
        if 0 <= num <= 10:
            return num
    except (ValueError, TypeError):
        pass
    return np.nan

# Get BPI columns using the defined item lists
bpi_interference_cols = [col for col in df_excel_valid.columns if col in BPI_INTERFERENCE_ITEMS]
bpi_intensity_cols = [col for col in df_excel_valid.columns if col in BPI_INTENSITY_ITEMS]

print(f'Found {len(bpi_interference_cols)} BPI Interference items: {bpi_interference_cols}')
print(f'Found {len(bpi_intensity_cols)} BPI Intensity items: {bpi_intensity_cols}')

# Convert BPI responses
for col in bpi_interference_cols + bpi_intensity_cols:
    df_excel_valid[col] = pd.to_numeric(df_excel_valid[col], errors='coerce')
    # Clip to valid range
    df_excel_valid[col] = df_excel_valid[col].clip(0, 10)

# Calculate BPI means
if len(bpi_interference_cols) > 0:
    df_excel_valid['bpi_interference_mean'] = df_excel_valid[bpi_interference_cols].mean(axis=1, skipna=True)
    print(f'\n✓ BPI Interference mean computed')
    print(f'  Mean={df_excel_valid["bpi_interference_mean"].mean():.2f}, SD={df_excel_valid["bpi_interference_mean"].std():.2f}')
    print(f'  Valid responses: {df_excel_valid["bpi_interference_mean"].notna().sum()} / {len(df_excel_valid)}')

if len(bpi_intensity_cols) > 0:
    df_excel_valid['bpi_intensity_mean'] = df_excel_valid[bpi_intensity_cols].mean(axis=1, skipna=True)
    print(f'✓ BPI Intensity mean computed')
    print(f'  Mean={df_excel_valid["bpi_intensity_mean"].mean():.2f}, SD={df_excel_valid["bpi_intensity_mean"].std():.2f}')

if len(bpi_interference_cols) > 0 and len(bpi_intensity_cols) > 0:
    df_excel_valid['bpi_total_mean'] = df_excel_valid[bpi_interference_cols + bpi_intensity_cols].mean(axis=1, skipna=True)
    print(f'✓ BPI Total mean computed')
    print(f'  Mean={df_excel_valid["bpi_total_mean"].mean():.2f}, SD={df_excel_valid["bpi_total_mean"].std():.2f}')

In [ ]:
# ============================================================================
# TSK-11SV (Tampa Scale of Kinesiophobia) Processing
# Scale: 1=Totalmente en desacuerdo, 2=En desacuerdo, 3=De acuerdo, 4=Totalmente de acuerdo
# ============================================================================
print('\n' + '='*80)
print('TSK-11SV (Tampa Scale of Kinesiophobia) - Processing')
print('='*80)

# TSK response mapping
TSK_RESPONSE_MAPPING = {
    'Totalmente en desacuerdo': 1,
    'En desacuerdo': 2,
    'De acuerdo': 3,
    'Totalmente de acuerdo': 4,
    '1': 1, '2': 2, '3': 3, '4': 4,
}

def convert_tsk_response(value):
    """Convert TSK text response to numeric (1-4)"""
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    
    # Try direct mapping
    if value_str in TSK_RESPONSE_MAPPING:
        return TSK_RESPONSE_MAPPING[value_str]
    
    # Try numeric conversion
    try:
        num = int(value_str)
        if 1 <= num <= 4:
            return num
    except (ValueError, TypeError):
        pass
    return np.nan

# Get TSK item columns (tsk_01 to tsk_11) using the defined item list
tsk_item_cols = [col for col in df_excel_valid.columns if col in TSK_ITEMS]
print(f'Found {len(tsk_item_cols)} TSK items: {tsk_item_cols}')

# Convert TSK responses
for col in tsk_item_cols:
    df_excel_valid[col] = df_excel_valid[col].apply(convert_tsk_response)

# Calculate TSK total
if len(tsk_item_cols) >= 11:
    df_excel_valid['tsk_total'] = df_excel_valid[tsk_item_cols].sum(axis=1, min_count=11)
    
    print(f'\n✓ TSK total score computed')
    print(f'  Total score: Mean={df_excel_valid["tsk_total"].mean():.2f}, SD={df_excel_valid["tsk_total"].std():.2f}')
    print(f'  Range: [{df_excel_valid["tsk_total"].min():.0f}, {df_excel_valid["tsk_total"].max():.0f}]')
    print(f'  Valid responses: {df_excel_valid["tsk_total"].notna().sum()} / {len(df_excel_valid)}')
else:
    print(f'\n⚠️  Insufficient TSK items found ({len(tsk_item_cols)}/11)')

print('\n' + '='*80)
print('✓ All questionnaire data processed and validated')
print('='*80)

### 2.4 Create Master Patient Data Tables

Save processed data to database in 4 normalized tables for efficient analyses

In [ ]:
# ============================================================================
# Create 4 Normalized Patient Data Tables
# ============================================================================
import hashlib

# Define narrative column name (after mapping)
NARRATIVE_COLUMN = 'narrative_text'

print('='*80)
print('Creating 4 Normalized Patient Data Tables')
print('='*80)

# Compute narrative hash and word count
def normalize_and_hash_narrative(text):
    """Normalize text and compute SHA-256 hash"""
    if pd.isna(text):
        return '', np.nan, 0
    
    # Normalize
    normalized = str(text).strip()
    normalized = re.sub(r'\s+', ' ', normalized)
    
    # Hash
    narrative_hash = hashlib.sha256(normalized.encode('utf-8')).hexdigest()
    
    # Word count
    word_count = len(normalized.split()) if normalized else 0
    
    return normalized, narrative_hash, word_count

# Apply to narrative text
if NARRATIVE_COLUMN in df_excel_valid.columns:
    result = df_excel_valid[NARRATIVE_COLUMN].apply(normalize_and_hash_narrative)
    df_excel_valid['narrative_normalized'] = result.apply(lambda x: x[0])
    df_excel_valid['narrative_hash'] = result.apply(lambda x: x[1])
    df_excel_valid['word_count'] = result.apply(lambda x: x[2])
    
    print(f'✓ Computed narrative_hash and word_count for {len(df_excel_valid)} participants')
    print(f'  Word count range: [{df_excel_valid["word_count"].min():.0f}, {df_excel_valid["word_count"].max():.0f}]')
    print(f'  Mean word count: {df_excel_valid["word_count"].mean():.1f}')
    print()
else:
    print(f'⚠️  Narrative column "{NARRATIVE_COLUMN}" not found')

# ============================================================================
# TABLE 1: Demographics & Narrative (Master Reference)
# ============================================================================
print('-'*80)
print('Table 1: real_patient_demographics')
print('-'*80)

demographics_cols = ['participant_id', 'narrative_hash', 'narrative_text', 'word_count']

# Add demographics
for col in ['age', 'gender', 'marital_status', 'education_level', 
            'country_residence', 'country_birth', 'employment_status']:
    if col in df_excel_valid.columns:
        demographics_cols.append(col)

# Add pain history
for col in ['years_with_pain', 'years_since_diagnosis', 'pain_cause_primary', 'pain_location_zones']:
    if col in df_excel_valid.columns:
        demographics_cols.append(col)

# Filter to existing columns
demographics_cols = [col for col in demographics_cols if col in df_excel_valid.columns]

df_demographics = df_excel_valid[demographics_cols].copy()

# Save to database
df_demographics.to_sql(
    'real_patient_demographics',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_demographics)} rows, {len(df_demographics.columns)} columns')
print(f'  Columns: {list(df_demographics.columns)}')

# ============================================================================
# TABLE 2: PCS Questionnaire Data
# ============================================================================
print()
print('-'*80)
print('Table 2: real_patient_pcs')
print('-'*80)

pcs_cols = ['participant_id']

# Add PCS items
pcs_cols += [col for col in PCS_ITEMS if col in df_excel_valid.columns]

# Add PCS computed scores
for col in ['pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness']:
    if col in df_excel_valid.columns:
        pcs_cols.append(col)

df_pcs = df_excel_valid[pcs_cols].copy()

# Save to database
df_pcs.to_sql(
    'real_patient_pcs',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_pcs)} rows, {len(df_pcs.columns)} columns')
print(f'  Items: {len([c for c in df_pcs.columns if c in PCS_ITEMS])} PCS items')
print(f'  Scores: pcs_total, pcs_rumination, pcs_magnification, pcs_helplessness')

# ============================================================================
# TABLE 3: BPI Questionnaire Data
# ============================================================================
print()
print('-'*80)
print('Table 3: real_patient_bpi')
print('-'*80)

bpi_cols = ['participant_id']

# Add BPI items
bpi_cols += [col for col in BPI_INTERFERENCE_ITEMS + BPI_INTENSITY_ITEMS if col in df_excel_valid.columns]

# Add BPI computed scores
for col in ['bpi_interference_mean', 'bpi_intensity_mean', 'bpi_total_mean']:
    if col in df_excel_valid.columns:
        bpi_cols.append(col)

df_bpi = df_excel_valid[bpi_cols].copy()

# Save to database
df_bpi.to_sql(
    'real_patient_bpi',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_bpi)} rows, {len(df_bpi.columns)} columns')
print(f'  Interference items: {len([c for c in df_bpi.columns if c in BPI_INTERFERENCE_ITEMS])}')
print(f'  Intensity items: {len([c for c in df_bpi.columns if c in BPI_INTENSITY_ITEMS])}')
print(f'  Scores: bpi_interference_mean, bpi_intensity_mean, bpi_total_mean')

# ============================================================================
# TABLE 4: TSK Questionnaire Data
# ============================================================================
print()
print('-'*80)
print('Table 4: real_patient_tsk')
print('-'*80)

tsk_cols = ['participant_id']

# Add TSK items
tsk_cols += [col for col in TSK_ITEMS if col in df_excel_valid.columns]

# Add TSK computed score
if 'tsk_total' in df_excel_valid.columns:
    tsk_cols.append('tsk_total')

df_tsk = df_excel_valid[tsk_cols].copy()

# Save to database
df_tsk.to_sql(
    'real_patient_tsk',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_tsk)} rows, {len(df_tsk.columns)} columns')
print(f'  Items: {len([c for c in df_tsk.columns if c in TSK_ITEMS])} TSK items')
print(f'  Scores: tsk_total')

# ============================================================================
# Verify All Tables
# ============================================================================
print()
print('='*80)
print('DATABASE VERIFICATION')
print('='*80)

tables_created = [
    'real_patient_demographics',
    'real_patient_pcs',
    'real_patient_bpi',
    'real_patient_tsk'
]

with Session(engine) as session:
    for table in tables_created:
        count = session.exec(
            text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table}')
        ).first()
        print(f'✓ {NEW_SCHEMA}.{table}: {count} rows')

print()
print('='*80)
print('✓ All 4 patient data tables created successfully!')
print('='*80)
print()
print('Table Relationships:')
print('  - All tables linked by: participant_id')
print('  - Experiments linked by: narrative_hash (in demographics table)')
print()
print('Example JOIN query:')
print('  SELECT d.participant_id, d.age, d.gender, p.pcs_total, b.bpi_total_mean, t.tsk_total')
print('  FROM real_patient_demographics d')
print('  LEFT JOIN real_patient_pcs p ON d.participant_id = p.participant_id')
print('  LEFT JOIN real_patient_bpi b ON d.participant_id = b.participant_id')
print('  LEFT JOIN real_patient_tsk t ON d.participant_id = t.participant_id')
print('  WHERE d.age > 30;')

### 2.5 Example Queries for Common Analyses

How to use the 4 normalized tables for typical research questions

In [ ]:
# ============================================================================
# Example Queries for Common Research Analyses
# ============================================================================

print('='*80)
print('EXAMPLE QUERIES FOR COMMON ANALYSES')
print('='*80)

# Example 1: Get all questionnaire data for a specific participant
print('\n' + '-'*80)
print('Example 1: Get complete data for participant ID = 1')
print('-'*80)

query1 = f"""
SELECT 
    d.participant_id,
    d.age,
    d.gender,
    d.narrative_hash,
    d.word_count,
    p.pcs_total,
    p.pcs_rumination,
    p.pcs_magnification,
    p.pcs_helplessness,
    b.bpi_interference_mean,
    b.bpi_intensity_mean,
    b.bpi_total_mean,
    t.tsk_total
FROM {NEW_SCHEMA}.real_patient_demographics d
LEFT JOIN {NEW_SCHEMA}.real_patient_pcs p ON d.participant_id = p.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_bpi b ON d.participant_id = b.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_tsk t ON d.participant_id = t.participant_id
WHERE d.participant_id = 1
"""

with Session(engine) as session:
    result = session.exec(text(query1)).first()
    if result:
        print(f'✓ Participant {result[0]}: Age={result[1]}, Gender={result[2]}')
        print(f'  PCS Total={result[5]:.0f}, BPI Mean={result[9]:.2f}, TSK Total={result[12]:.0f}')

# Example 2: Get descriptive statistics for all questionnaires
print('\n' + '-'*80)
print('Example 2: Descriptive statistics for all questionnaires')
print('-'*80)

query2 = f"""
SELECT 
    COUNT(*) as n_participants,
    AVG(p.pcs_total) as pcs_mean,
    STDDEV(p.pcs_total) as pcs_sd,
    AVG(b.bpi_total_mean) as bpi_mean,
    STDDEV(b.bpi_total_mean) as bpi_sd,
    AVG(t.tsk_total) as tsk_mean,
    STDDEV(t.tsk_total) as tsk_sd
FROM {NEW_SCHEMA}.real_patient_demographics d
LEFT JOIN {NEW_SCHEMA}.real_patient_pcs p ON d.participant_id = p.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_bpi b ON d.participant_id = b.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_tsk t ON d.participant_id = t.participant_id
"""

with Session(engine) as session:
    result = session.exec(text(query2)).first()
    if result:
        print(f'✓ N={result[0]} participants')
        print(f'  PCS: Mean={result[1]:.2f}, SD={result[2]:.2f}')
        print(f'  BPI: Mean={result[3]:.2f}, SD={result[4]:.2f}')
        print(f'  TSK: Mean={result[5]:.2f}, SD={result[6]:.2f}')

# Example 3: Link to synthetic LLM data via narrative_hash
print('\n' + '-'*80)
print('Example 3: Compare real vs synthetic PCS scores (first 5 participants)')
print('-'*80)

# Note: This assumes synthetic data exists in the database
# We'll create a conceptual example query
query3_example = f"""
-- This query will work once synthetic data is loaded
SELECT 
    d.participant_id,
    d.narrative_hash,
    r_pcs.pcs_total as real_pcs_total,
    s_pcs.pcs_total as synthetic_pcs_total,
    (s_pcs.pcs_total - r_pcs.pcs_total) as difference
FROM {NEW_SCHEMA}.real_patient_demographics d
INNER JOIN {NEW_SCHEMA}.real_patient_pcs r_pcs 
    ON d.participant_id = r_pcs.participant_id
INNER JOIN {NEW_SCHEMA}.synthetic_patient_pcs s_pcs 
    ON d.narrative_hash = s_pcs.narrative_hash
ORDER BY d.participant_id
LIMIT 5
"""

print('Query template (for when synthetic data is loaded):')
print(query3_example)

# Example 4: Filter by demographics
print('\n' + '-'*80)
print('Example 4: Get questionnaire data for participants aged 40-60')
print('-'*80)

query4 = f"""
SELECT 
    d.participant_id,
    d.age,
    d.gender,
    p.pcs_total,
    b.bpi_total_mean,
    t.tsk_total
FROM {NEW_SCHEMA}.real_patient_demographics d
LEFT JOIN {NEW_SCHEMA}.real_patient_pcs p ON d.participant_id = p.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_bpi b ON d.participant_id = b.participant_id
LEFT JOIN {NEW_SCHEMA}.real_patient_tsk t ON d.participant_id = t.participant_id
WHERE d.age BETWEEN 40 AND 60
ORDER BY d.age
LIMIT 5
"""

with Session(engine) as session:
    results = session.exec(text(query4)).all()
    if results:
        print(f'✓ Found {len(results)} participants (showing first 5)')
        for r in results[:5]:
            print(f'  ID={r[0]}, Age={r[1]}, Gender={r[2]}, PCS={r[3]:.0f}, BPI={r[4]:.2f}, TSK={r[5]:.0f}')

print('\n' + '='*80)
print('✓ Example queries completed')
print('='*80)

---

## Summary: Complete Database Schema Structure

### Schema: `pain_narratives_acm_202512`

#### Patient Data Tables (4 tables)

**1. real_patient_demographics** (Master Reference)
- **Primary Key**: `participant_id`
- **Unique Key**: `narrative_hash` (for linking experiments)
- **Contains**: Demographics, narrative text, pain history
- **Links to**: All other tables via participant_id or narrative_hash

**2. real_patient_pcs** (Pain Catastrophizing Scale)
- **Foreign Key**: `participant_id`
- **Items**: `pcs_01` to `pcs_13` (13 items, scale 0-4)
- **Scores**: `pcs_total`, `pcs_rumination`, `pcs_magnification`, `pcs_helplessness`

**3. real_patient_bpi** (Brief Pain Inventory)
- **Foreign Key**: `participant_id`
- **Interference**: `bpiq11` to `bpiq17` (7 items, scale 0-10)
- **Intensity**: `bpiq28`, `bpiq39`, `bpiq410`, `bpiq511` (4 items, scale 0-10)
- **Scores**: `bpi_interference_mean`, `bpi_intensity_mean`, `bpi_total_mean`

**4. real_patient_tsk** (Tampa Scale of Kinesiophobia)
- **Foreign Key**: `participant_id`
- **Items**: `tsk_01` to `tsk_11` (11 items, scale 1-4)
- **Scores**: `tsk_total`

#### Expert Users Table (1 table)

**5. expert_users**
- **Foreign Key**: `user_id` (links to pain_narratives_app.users)
- **Assigned Narratives**: `assigned_narrative_01/02/03` + corresponding hashes
- **Status**: `evaluation_completed` boolean

#### Expert UI Feedback Tables (2 tables)

**6. expert_dimension_evaluation** (Expert ratings of LLM dimension scores)
- **Foreign Keys**: `participant_id`, `user_id`, `narrative_hash`
- **Dimensions**: Severidad del dolor, Discapacidad
- **Ratings**: `{dimension}_score_alignment`, `{dimension}_explanation_alignment`, `{dimension}_usage_intent`

**7. expert_questionnaire_feedback** (Expert ratings of LLM questionnaire results)
- **Foreign Keys**: `participant_id`, `user_id`, `narrative_hash`
- **Questionnaires**: PCS, BPI-IS, TSK-11SV
- **Ratings**: `authenticity_rating`, `reasoning_adequacy_rating`

#### Key Linkages

```
participant_id → Links patient data tables (demographics ↔ pcs ↔ bpi ↔ tsk)
narrative_hash → Links across all data sources (patients ↔ experts ↔ experiments)
user_id → Links expert_users to expert_dimension_evaluation and expert_questionnaire_feedback
```

#### Next Steps
- ⏳ Load synthetic LLM questionnaire data (batch groups 35-36)
- ⏳ Perform reliability analyses (Cronbach's alpha, ICC, correlations)
- ⏳ Compare real patient scores vs LLM synthetic scores
- ⏳ Analyze expert agreement with LLM evaluations

## 5. Load Expert UI Dimension Evaluation Data

Load expert dimension evaluation feedback from the UI (groups 16-32) and normalize for analysis.

In [ ]:
# ============================================================================
# Load Expert UI Dimension Evaluation Data
# ============================================================================
print('='*80)
print('Loading Expert UI Dimension Evaluation Data')
print('='*80)

# Read the SQL query
with open(PROJECT_ROOT / 'sql' / 'experts_ui_master_dimension_evaluation.sql', 'r') as f:
    dim_eval_query = f.read()

# Execute the query
with Session(engine) as session:
    result = session.exec(text(dim_eval_query))
    dim_rows = result.fetchall()
    dim_columns = result.keys()

df_expert_dim_raw = pd.DataFrame(dim_rows, columns=dim_columns)

print(f'\n✓ Loaded {len(df_expert_dim_raw)} dimension evaluations')
print(f'  Unique users: {df_expert_dim_raw["user_id"].nunique()}')
print(f'  Unique narratives: {df_expert_dim_raw["narrative_id"].nunique()}')
print(f'  Unique narrative_hashes: {df_expert_dim_raw["narrative_hash"].nunique()}')
print(f'  Experiment groups: {sorted(df_expert_dim_raw["experiments_group_id"].unique())}')

# Show sample
print(f'\nSample data:')
print(df_expert_dim_raw[['user_id', 'narrative_id', 'narrative_hash', 'experiments_group_id']].head())

### 5.1 Normalize Dimension Feedback and Map to Patient Data

Extract dimension feedback ratings from JSON and link to patient demographics via narrative_hash.

In [ ]:
# ============================================================================
# Normalize Dimension Feedback JSON and Map to Patient Demographics
# ============================================================================
print('='*80)
print('Normalizing Dimension Feedback and Mapping to Patient Data')
print('='*80)

# Create mapping from narrative_hash to participant_id
hash_to_participant = dict(zip(
    df_demographics['narrative_hash'], 
    df_demographics['participant_id']
))
print(f'\n✓ Created hash->participant_id mapping with {len(hash_to_participant)} entries')

# IMPORTANT: Re-compute narrative_hash using SMART MATCHING
# Try standard normalization first, then fall back to quote-stripped version
def compute_hash_standard(narrative_text):
    """Standard normalization (whitespace only)"""
    if pd.isna(narrative_text) or not narrative_text:
        return None
    normalized = str(narrative_text).strip()
    normalized = re.sub(r'\s+', ' ', normalized)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

def compute_hash_stripped(narrative_text):
    """Strip leading/trailing quotes then normalize"""
    if pd.isna(narrative_text) or not narrative_text:
        return None
    normalized = str(narrative_text).strip()
    # Strip leading/trailing quotes
    if normalized.startswith('"'):
        normalized = normalized[1:]
    if normalized.endswith('"'):
        normalized = normalized[:-1]
    normalized = normalized.strip()
    normalized = re.sub(r'\s+', ' ', normalized)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

def smart_match_narrative(narrative_text, hash_lookup):
    """Try standard hash first, then fall back to stripped hash"""
    # Try standard normalization
    hash_std = compute_hash_standard(narrative_text)
    if hash_std and hash_std in hash_lookup:
        return hash_std, hash_lookup[hash_std], 'standard'
    
    # Try stripped quotes
    hash_stripped = compute_hash_stripped(narrative_text)
    if hash_stripped and hash_stripped in hash_lookup:
        return hash_stripped, hash_lookup[hash_stripped], 'stripped'
    
    # No match found - return standard hash anyway
    return hash_std, None, 'no_match'

# Apply smart matching
match_results = df_expert_dim_raw['narrative'].apply(
    lambda x: smart_match_narrative(x, hash_to_participant)
)

df_expert_dim_raw['narrative_hash_corrected'] = match_results.apply(lambda x: x[0])
df_expert_dim_raw['participant_id_matched'] = match_results.apply(lambda x: x[1])
df_expert_dim_raw['match_method'] = match_results.apply(lambda x: x[2])

# Report matching results
match_counts = df_expert_dim_raw['match_method'].value_counts()
print(f'\n✓ Smart matching results:')
for method, count in match_counts.items():
    print(f'  - {method}: {count}')

# Extract dimension feedback ratings
def extract_dimension_ratings(dim_feedback):
    """Extract ratings from dimension_feedback JSON"""
    if not dim_feedback or not isinstance(dim_feedback, dict):
        return {}
    
    ratings = {}
    for dimension_name, dimension_data in dim_feedback.items():
        if isinstance(dimension_data, dict):
            # Normalize dimension name
            dim_key = dimension_name.lower().replace(' del dolor', '').replace(' ', '_')
            
            # Extract ratings
            for rating_type in ['score_alignment', 'explanation_alignment', 'usage_intent']:
                col_name = f'{dim_key}_{rating_type}'
                ratings[col_name] = dimension_data.get(rating_type, None)
    
    return ratings

# Apply extraction
dim_ratings = df_expert_dim_raw['dimension_feedback'].apply(extract_dimension_ratings)
df_dim_ratings = pd.DataFrame(dim_ratings.tolist())

# Combine with original data (use corrected hash and matched participant_id)
df_expert_dim = pd.concat([
    df_expert_dim_raw[['experiments_group_id', 'experiment_id', 'user_id', 
                       'narrative_id', 'word_count']],
    df_expert_dim_raw[['narrative_hash_corrected']].rename(columns={'narrative_hash_corrected': 'narrative_hash'}),
    df_expert_dim_raw[['participant_id_matched']].rename(columns={'participant_id_matched': 'participant_id'}),
    df_dim_ratings
], axis=1)

# Reorder columns
cols_order = ['participant_id', 'narrative_id', 'narrative_hash', 
              'experiments_group_id', 'experiment_id', 'user_id', 'word_count']
cols_order += [c for c in df_expert_dim.columns if c not in cols_order]
df_expert_dim = df_expert_dim[cols_order]

print(f'\n✓ Extracted dimension ratings into {len(df_dim_ratings.columns)} columns:')
for col in df_dim_ratings.columns:
    print(f'  - {col}')

print(f'\n✓ Mapped {df_expert_dim["participant_id"].notna().sum()}/{len(df_expert_dim)} rows to patient data')

# Show unmatched narratives for transparency
unmatched_count = df_expert_dim['participant_id'].isna().sum()
if unmatched_count > 0:
    print(f'\n⚠️  {unmatched_count} rows could not be matched (test/placeholder narratives)')
    unmatched = df_expert_dim[df_expert_dim['participant_id'].isna()][['narrative_id', 'word_count']].drop_duplicates()
    print(f'   Unmatched: {unmatched.to_dict("records")}')

# Show rating value distribution
print(f'\nRating Values Distribution (sample - severidad_score_alignment):')
if 'severidad_score_alignment' in df_expert_dim.columns:
    for val, count in df_expert_dim['severidad_score_alignment'].value_counts().items():
        print(f'  - {val}: {count}')

print(f'\nSample normalized data:')
print(df_expert_dim[['participant_id', 'user_id', 'severidad_score_alignment', 'discapacidad_score_alignment']].head())

### 5.2 Save Expert Dimension Evaluation Table to Database

In [ ]:
# ============================================================================
# Save Expert Dimension Evaluation Table to Database
# ============================================================================
print('='*80)
print('Saving Expert Dimension Evaluation Table')
print('='*80)

# IMPORTANT: Filter out unmatched records (test/placeholder narratives)
# Only keep records that successfully linked to real patient data
df_expert_dim_filtered = df_expert_dim[df_expert_dim['participant_id'].notna()].copy()

unmatched_count = len(df_expert_dim) - len(df_expert_dim_filtered)
if unmatched_count > 0:
    print(f'\n⚠️  Filtered out {unmatched_count} unmatched records (test/placeholder data)')
    unmatched_ids = df_expert_dim[df_expert_dim['participant_id'].isna()]['narrative_id'].unique()
    print(f'   Excluded narrative_ids: {sorted(unmatched_ids)}')

# Save to database (only complete data)
table_name = 'expert_dimension_evaluation'

df_expert_dim_filtered.to_sql(
    table_name,
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'\n✓ Saved to {NEW_SCHEMA}.{table_name}')
print(f'  Rows: {len(df_expert_dim_filtered)} (filtered from {len(df_expert_dim)})')
print(f'  Columns: {len(df_expert_dim_filtered.columns)}')
print(f'\nColumns:')
for col in df_expert_dim_filtered.columns:
    print(f'  - {col}')

# Verify
with Session(engine) as session:
    count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table_name}')).first()
    print(f'\n✓ Verified: {count[0]} rows in database')
    
    # Verify no NULL participant_ids
    null_count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table_name} WHERE participant_id IS NULL')).first()
    print(f'✓ NULL participant_ids: {null_count[0]} (should be 0)')
    
print(f'\n✓ Database contains only complete, validated expert dimension evaluations')

## 6. Load Expert UI Questionnaire Feedback Data

Load expert questionnaire feedback from the UI (groups 16-32). This contains authenticity and reasoning adequacy ratings for PCS, BPI-IS, and TSK-11SV questionnaires.

In [ ]:
# ============================================================================
# Load Expert UI Questionnaire Feedback Data
# ============================================================================
print('='*80)
print('Loading Expert UI Questionnaire Feedback Data')
print('='*80)

# Read the SQL query
with open(PROJECT_ROOT / 'sql' / 'experts_ui_master_questionnaires.sql', 'r') as f:
    quest_query = f.read()

# Execute the query
with Session(engine) as session:
    result = session.exec(text(quest_query))
    quest_rows = result.fetchall()
    quest_columns = result.keys()

df_expert_quest_raw = pd.DataFrame(quest_rows, columns=quest_columns)

print(f'\n✓ Loaded {len(df_expert_quest_raw)} questionnaire feedback records')
print(f'  Unique users: {df_expert_quest_raw["user_id"].nunique()}')
print(f'  Unique narratives: {df_expert_quest_raw["narrative_id"].nunique()}')
print(f'  Unique narrative_hashes: {df_expert_quest_raw["narrative_hash"].nunique()}')
print(f'  Experiment groups: {sorted(df_expert_quest_raw["experiments_group_id"].unique())}')

print(f'\nQuestionnaire Types:')
for qname, count in df_expert_quest_raw['questionnaire_name'].value_counts().items():
    print(f'  - {qname}: {count}')

print(f'\nAuthenticity Ratings:')
for rating, count in df_expert_quest_raw['authenticity_rating'].value_counts().items():
    print(f'  - {rating}: {count}')

print(f'\nReasoning Adequacy Ratings:')
for rating, count in df_expert_quest_raw['reasoning_adequacy_rating'].value_counts().items():
    print(f'  - {rating}: {count}')

### 6.1 Normalize and Map Questionnaire Feedback to Patient Data

In [ ]:
# ============================================================================
# Normalize Questionnaire Feedback and Map to Patient Data
# ============================================================================
print('='*80)
print('Normalizing Questionnaire Feedback and Mapping to Patient Data')
print('='*80)

# Apply smart matching (using functions defined in previous cell)
match_results = df_expert_quest_raw['narrative'].apply(
    lambda x: smart_match_narrative(x, hash_to_participant)
)

df_expert_quest_raw['narrative_hash_corrected'] = match_results.apply(lambda x: x[0])
df_expert_quest_raw['participant_id_matched'] = match_results.apply(lambda x: x[1])
df_expert_quest_raw['match_method'] = match_results.apply(lambda x: x[2])

# Report matching results
match_counts = df_expert_quest_raw['match_method'].value_counts()
print(f'\n✓ Smart matching results:')
for method, count in match_counts.items():
    print(f'  - {method}: {count}')

# Select and normalize columns (use corrected hash and matched participant_id)
df_expert_quest = df_expert_quest_raw[[
    'experiments_group_id', 'user_id', 'narrative_id', 'narrative_hash_corrected',
    'participant_id_matched', 'word_count', 'questionnaire_id', 'questionnaire_name',
    'authenticity_rating', 'reasoning_adequacy_rating'
]].copy()

# Rename columns
df_expert_quest = df_expert_quest.rename(columns={
    'narrative_hash_corrected': 'narrative_hash',
    'participant_id_matched': 'participant_id'
})

# Reorder columns
cols_order = ['participant_id', 'narrative_id', 'narrative_hash',
              'experiments_group_id', 'user_id', 'questionnaire_id', 'questionnaire_name',
              'authenticity_rating', 'reasoning_adequacy_rating', 'word_count']
df_expert_quest = df_expert_quest[cols_order]

print(f'\n✓ Normalized {len(df_expert_quest)} questionnaire feedback records')
print(f'✓ Mapped {df_expert_quest["participant_id"].notna().sum()}/{len(df_expert_quest)} rows to patient data')

# Show unmatched narratives for transparency
unmatched_count = df_expert_quest['participant_id'].isna().sum()
if unmatched_count > 0:
    print(f'\n⚠️  {unmatched_count} rows could not be matched (test/placeholder narratives)')
    unmatched = df_expert_quest[df_expert_quest['participant_id'].isna()][['narrative_id', 'word_count']].drop_duplicates()
    print(f'   Unmatched: {unmatched.to_dict("records")}')

print(f'\nData structure:')
print(f'  Columns: {list(df_expert_quest.columns)}')

print(f'\nSample data:')
print(df_expert_quest[['participant_id', 'user_id', 'questionnaire_name', 'authenticity_rating', 'reasoning_adequacy_rating']].head())

### 6.2 Save Expert Questionnaire Feedback Table to Database

In [ ]:
# ============================================================================
# Save Expert Questionnaire Feedback Table to Database
# ============================================================================
print('='*80)
print('Saving Expert Questionnaire Feedback Table')
print('='*80)

# IMPORTANT: Filter out unmatched records (test/placeholder narratives)
# Only keep records that successfully linked to real patient data
df_expert_quest_filtered = df_expert_quest[df_expert_quest['participant_id'].notna()].copy()

unmatched_count = len(df_expert_quest) - len(df_expert_quest_filtered)
if unmatched_count > 0:
    print(f'\n⚠️  Filtered out {unmatched_count} unmatched records (test/placeholder data)')
    unmatched_ids = df_expert_quest[df_expert_quest['participant_id'].isna()]['narrative_id'].unique()
    print(f'   Excluded narrative_ids: {sorted(unmatched_ids)}')

# Save to database (only complete data)
table_name = 'expert_questionnaire_feedback'

df_expert_quest_filtered.to_sql(
    table_name,
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'\n✓ Saved to {NEW_SCHEMA}.{table_name}')
print(f'  Rows: {len(df_expert_quest_filtered)} (filtered from {len(df_expert_quest)})')
print(f'  Columns: {len(df_expert_quest_filtered.columns)}')
print(f'\nColumns:')
for col in df_expert_quest_filtered.columns:
    print(f'  - {col}')

# Verify
with Session(engine) as session:
    count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table_name}')).first()
    print(f'\n✓ Verified: {count[0]} rows in database')
    
    # Verify no NULL participant_ids
    null_count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table_name} WHERE participant_id IS NULL')).first()
    print(f'✓ NULL participant_ids: {null_count[0]} (should be 0)')

# Show breakdown by questionnaire type
print(f'\n✓ Questionnaire breakdown:')
for qname, count in df_expert_quest_filtered['questionnaire_name'].value_counts().items():
    print(f'  - {qname}: {count} evaluations')
    
print(f'\n✓ Database contains only complete, validated expert questionnaire feedback')

## 7. Final Schema Summary and Verification

Verify all tables in the schema and summarize the complete data structure.

In [ ]:
# ============================================================================
# Final Schema Summary and Verification
# ============================================================================
print('='*80)
print(f'COMPLETE SCHEMA SUMMARY: {NEW_SCHEMA}')
print('='*80)

with Session(engine) as session:
    # Get all tables in the schema
    tables_query = text(f"""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = '{NEW_SCHEMA}'
        ORDER BY table_name
    """)
    tables = session.exec(tables_query).all()
    
    print(f'\nTables in schema ({len(tables)}):')
    print('-'*80)
    
    for table in tables:
        table_name = table[0]
        count_query = text(f"SELECT COUNT(*) FROM {NEW_SCHEMA}.{table_name}")
        count = session.exec(count_query).first()
        
        cols_query = text(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = '{NEW_SCHEMA}' AND table_name = '{table_name}'
            ORDER BY ordinal_position
        """)
        cols = session.exec(cols_query).all()
        
        print(f'\n{table_name}')
        print(f'  Rows: {count[0]}')
        print(f'  Columns ({len(cols)}): {", ".join([c[0] for c in cols[:5]])}{"..." if len(cols) > 5 else ""}')

print('\n' + '='*80)
print('KEY RELATIONSHIPS')
print('='*80)
print('''
PATIENT DATA (152 participants):
  └── real_patient_demographics (master table)
      ├── real_patient_pcs (PCS questionnaire - 13 items)
      ├── real_patient_bpi (BPI questionnaire - 11 items)
      └── real_patient_tsk (TSK questionnaire - 11 items)
      
      LINKED BY: participant_id, narrative_hash

EXPERT USERS (17 evaluators):
  └── expert_users
      └── Links to real_patient_demographics via assigned_narrative_hash_XX

EXPERT UI FEEDBACK (groups 16-32):
  ├── expert_dimension_evaluation
  │   └── Expert ratings of LLM dimension scores (Severidad, Discapacidad)
  │   └── ✓ All records linked to real patient data
  │
  └── expert_questionnaire_feedback
      └── Expert ratings of LLM questionnaire results (PCS, BPI-IS, TSK-11SV)
      └── ✓ All records linked to real patient data

LLM SYNTHETIC DATA (batch groups 35-36):
  ├── llm_dimension_evaluation
  │   └── LLM scores for Severidad del dolor & Discapacidad
  │
  ├── llm_pcs_results
  │   └── LLM PCS questionnaire responses (13 items + subscales)
  │
  ├── llm_bpi_results
  │   └── LLM BPI-IS questionnaire responses (11 items + subscales)
  │
  └── llm_tsk_results
      └── LLM TSK-11SV questionnaire responses (11 items)
      
      LINKED BY: participant_id, experiment_id, run_number

DATA QUALITY:
  ✓ All expert feedback tables contain ONLY complete data
  ✓ All LLM synthetic data tables contain ONLY complete data
  ✓ All records successfully linked to real patient narratives
  ✓ Test/placeholder narratives filtered out during processing

COMPARISON QUERIES:
  -- Compare real vs LLM PCS scores
  SELECT 
      r.participant_id,
      r.pcs_total as real_pcs,
      l.pcs_total as llm_pcs,
      (l.pcs_total - r.pcs_total) as difference
  FROM real_patient_pcs r
  JOIN llm_pcs_results l ON r.participant_id = l.participant_id;
  
  -- Get expert feedback on LLM dimension scores
  SELECT 
      d.participant_id,
      l.severidad_score as llm_severidad,
      e.severidad_score_alignment as expert_alignment
  FROM real_patient_demographics d
  JOIN llm_dimension_evaluation l ON d.participant_id = l.participant_id
  LEFT JOIN expert_dimension_evaluation e ON d.participant_id = e.participant_id;
''')

print('='*80)
print('✓ All tables created and verified!')
print('✓ Database contains only complete, validated data')
print('✓ Ready for analysis and future batch runs')
print('='*80)

## 8. LLM Synthetic Data Migration (Batch Processing Groups 35-36)

Migrate LLM-generated questionnaire and dimension evaluation data from batch processing experiments.

**Data Source**: Experiment groups 35 (145 narratives) and 36 (7 re-processed narratives)
- Model: GPT-5
- User: batch_processor (user_id=37)
- Evaluations: Dimension scores + PCS + BPI-IS + TSK-11SV

**Tables to Create**:
1. `llm_experiment_runs` - Metadata about each batch run
2. `llm_dimension_evaluation` - Severidad del dolor & Discapacidad scores
3. `llm_pcs_results` - Pain Catastrophizing Scale responses
4. `llm_bpi_results` - Brief Pain Inventory responses
5. `llm_tsk_results` - Tampa Scale of Kinesiophobia responses

In [ ]:
# ============================================================================
# 8.1 Load LLM Batch Processing Data
# ============================================================================
print('='*80)
print('Loading LLM Batch Processing Data (Groups 35-36)')
print('='*80)

# Define batch experiment groups
BATCH_GROUPS = [35, 36]

# Load experiment groups metadata
groups_query = text("""
    SELECT experiments_group_id, description, owner_id, concluded, created
    FROM pain_narratives_app.experiments_groups
    WHERE experiments_group_id IN (35, 36)
    ORDER BY experiments_group_id
""")

with Session(engine) as session:
    groups_df = pd.DataFrame(session.exec(groups_query).fetchall(),
                            columns=['experiments_group_id', 'description', 'owner_id', 'concluded', 'created'])

print('\nExperiment Groups:')
print(groups_df.to_string())

# Load all experiments with narratives
experiments_query = text("""
    SELECT 
        el.experiment_id,
        el.experiments_group_id,
        el.user_id,
        el.narrative_id,
        el.model,
        el.exp_type,
        el.succeeded,
        n.narrative,
        n.word_count
    FROM pain_narratives_app.experiments_list el
    JOIN pain_narratives_app.narratives n ON el.narrative_id = n.narrative_id
    WHERE el.experiments_group_id IN (35, 36)
    ORDER BY el.experiments_group_id, el.experiment_id
""")

with Session(engine) as session:
    experiments_df = pd.DataFrame(session.exec(experiments_query).fetchall(),
                                 columns=['experiment_id', 'experiments_group_id', 'user_id', 
                                         'narrative_id', 'model', 'exp_type', 'succeeded',
                                         'narrative', 'word_count'])

print(f'\n✓ Loaded {len(experiments_df)} experiments')
print(f'  Group 35: {len(experiments_df[experiments_df["experiments_group_id"] == 35])}')
print(f'  Group 36: {len(experiments_df[experiments_df["experiments_group_id"] == 36])}')
print(f'  Model: {experiments_df["model"].unique()}')
print(f'  Succeeded: {experiments_df["succeeded"].sum()}/{len(experiments_df)}')

In [ ]:
# ============================================================================
# 8.2 Link LLM Experiments to Patient Data
# ============================================================================
print('='*80)
print('Linking LLM Experiments to Patient Data via Narrative Hash')
print('='*80)

# Recompute narrative hash and link to patient data
# Using the same normalization as patient demographics
experiments_df['computed_hash'] = experiments_df['narrative'].apply(compute_hash_standard)
experiments_df['participant_id'] = experiments_df['computed_hash'].map(hash_to_participant)

matched = experiments_df['participant_id'].notna().sum()
print(f'\n✓ Matched {matched}/{len(experiments_df)} experiments to patient data ({100*matched/len(experiments_df):.1f}%)')

if experiments_df['participant_id'].isna().any():
    unmatched = experiments_df[experiments_df['participant_id'].isna()]
    print(f'\n⚠️  Unmatched experiments: {len(unmatched)}')
    print(unmatched[['experiment_id', 'narrative_id', 'word_count']].head())

# Create experiment to participant mapping
exp_to_participant = dict(zip(experiments_df['experiment_id'], experiments_df['participant_id']))
exp_to_group = dict(zip(experiments_df['experiment_id'], experiments_df['experiments_group_id']))
exp_to_model = dict(zip(experiments_df['experiment_id'], experiments_df['model']))

print(f'\n✓ Created experiment->participant mapping for {len(exp_to_participant)} experiments')

In [ ]:
# ============================================================================
# 8.3 Load and Process LLM Dimension Evaluations
# ============================================================================
print('='*80)
print('Loading LLM Dimension Evaluations')
print('='*80)

dim_query = text("""
    SELECT 
        er.experiment_id,
        er.experiments_group_id,
        er.narrative_id,
        er.result_type,
        er.result_json
    FROM pain_narratives_app.evaluation_results er
    WHERE er.experiments_group_id IN (35, 36)
      AND er.result_type = 'dimensions'
""")

with Session(engine) as session:
    dim_df = pd.DataFrame(session.exec(dim_query).fetchall(),
                         columns=['experiment_id', 'experiments_group_id', 'narrative_id', 
                                 'result_type', 'result_json'])

print(f'✓ Loaded {len(dim_df)} dimension evaluations')

# Extract dimension scores
def extract_dimension_scores(result_json):
    """Extract Severidad and Discapacidad scores from result_json"""
    if not result_json or not isinstance(result_json, dict):
        return {}
    
    scores = {}
    
    # Severidad del dolor
    if 'Severidad del dolor' in result_json:
        sev = result_json['Severidad del dolor']
        scores['severidad_score'] = sev.get('score') if isinstance(sev, dict) else None
        scores['severidad_explicacion'] = sev.get('explicacion') if isinstance(sev, dict) else None
    
    # Discapacidad
    if 'Discapacidad' in result_json:
        dis = result_json['Discapacidad']
        scores['discapacidad_score'] = dis.get('score') if isinstance(dis, dict) else None
        scores['discapacidad_explicacion'] = dis.get('explicacion') if isinstance(dis, dict) else None
    
    return scores

# Apply extraction
dim_scores = dim_df['result_json'].apply(extract_dimension_scores)
dim_scores_df = pd.DataFrame(dim_scores.tolist())

# Combine with experiment data
df_llm_dimensions = pd.concat([
    dim_df[['experiment_id', 'experiments_group_id', 'narrative_id']],
    dim_scores_df
], axis=1)

# Add participant_id and model
df_llm_dimensions['participant_id'] = df_llm_dimensions['experiment_id'].map(exp_to_participant)
df_llm_dimensions['model'] = df_llm_dimensions['experiment_id'].map(exp_to_model)

# Assign run_number based on experiment group
df_llm_dimensions['run_number'] = df_llm_dimensions['experiments_group_id'].map({35: 1, 36: 1})

# Filter to only matched records
df_llm_dimensions = df_llm_dimensions[df_llm_dimensions['participant_id'].notna()].copy()

# Reorder columns
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'severidad_score', 'severidad_explicacion', 
              'discapacidad_score', 'discapacidad_explicacion']
df_llm_dimensions = df_llm_dimensions[cols_order]

print(f'\n✓ Extracted dimension scores for {len(df_llm_dimensions)} records')
print(f'\nSeveridad del dolor scores:')
print(f'  Mean: {df_llm_dimensions["severidad_score"].mean():.2f}')
print(f'  Range: [{df_llm_dimensions["severidad_score"].min()}, {df_llm_dimensions["severidad_score"].max()}]')
print(f'\nDiscapacidad scores:')
print(f'  Mean: {df_llm_dimensions["discapacidad_score"].mean():.2f}')
print(f'  Range: [{df_llm_dimensions["discapacidad_score"].min()}, {df_llm_dimensions["discapacidad_score"].max()}]')

print(f'\nSample data:')
print(df_llm_dimensions[['participant_id', 'severidad_score', 'discapacidad_score', 'model']].head())

In [ ]:
# ============================================================================
# 8.4 Load and Process LLM PCS Questionnaire Results
# ============================================================================
print('='*80)
print('Loading LLM PCS Questionnaire Results')
print('='*80)

pcs_query = text("""
    SELECT 
        er.experiment_id,
        er.experiments_group_id,
        er.narrative_id,
        er.result_json
    FROM pain_narratives_app.evaluation_results er
    WHERE er.experiments_group_id IN (35, 36)
      AND er.result_type = 'PCS'
""")

with Session(engine) as session:
    pcs_df = pd.DataFrame(session.exec(pcs_query).fetchall(),
                         columns=['experiment_id', 'experiments_group_id', 'narrative_id', 'result_json'])

print(f'✓ Loaded {len(pcs_df)} PCS results')

# Extract PCS scores and item responses
def extract_pcs_data(result_json):
    """Extract PCS total score, subscales, and individual item scores"""
    if not result_json or not isinstance(result_json, dict):
        return {}
    
    data = {
        'pcs_total': result_json.get('total_score'),
    }
    
    # Subscales
    subscales = result_json.get('subscales', {})
    data['pcs_rumination'] = subscales.get('rumination')
    data['pcs_magnification'] = subscales.get('magnification')
    data['pcs_helplessness'] = subscales.get('helplessness')
    
    # Individual item scores (from raw_data.scores)
    raw_data = result_json.get('raw_data', {})
    scores = raw_data.get('scores', {})
    for i in range(1, 14):
        data[f'pcs_{str(i).zfill(2)}'] = scores.get(str(i))
    
    # Model reasoning
    data['model_reasoning'] = raw_data.get('model_reasoning')
    
    # Persona info
    persona = raw_data.get('persona', {})
    data['persona_name'] = persona.get('name')
    data['persona_traits'] = persona.get('traits')
    
    return data

# Apply extraction
pcs_data = pcs_df['result_json'].apply(extract_pcs_data)
pcs_data_df = pd.DataFrame(pcs_data.tolist())

# Combine with experiment data
df_llm_pcs = pd.concat([
    pcs_df[['experiment_id', 'experiments_group_id', 'narrative_id']],
    pcs_data_df
], axis=1)

# Add participant_id and model
df_llm_pcs['participant_id'] = df_llm_pcs['experiment_id'].map(exp_to_participant)
df_llm_pcs['model'] = df_llm_pcs['experiment_id'].map(exp_to_model)
df_llm_pcs['run_number'] = df_llm_pcs['experiments_group_id'].map({35: 1, 36: 1})

# Filter to only matched records
df_llm_pcs = df_llm_pcs[df_llm_pcs['participant_id'].notna()].copy()

# Reorder columns
item_cols = [f'pcs_{str(i).zfill(2)}' for i in range(1, 14)]
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness'] + item_cols + \
             ['persona_name', 'persona_traits', 'model_reasoning']
df_llm_pcs = df_llm_pcs[cols_order]

print(f'\n✓ Extracted PCS data for {len(df_llm_pcs)} records')
print(f'\nPCS Total Score:')
print(f'  Mean: {df_llm_pcs["pcs_total"].mean():.2f}')
print(f'  SD: {df_llm_pcs["pcs_total"].std():.2f}')
print(f'  Range: [{df_llm_pcs["pcs_total"].min()}, {df_llm_pcs["pcs_total"].max()}]')

print(f'\nSample data:')
print(df_llm_pcs[['participant_id', 'pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness']].head())

In [ ]:
# ============================================================================
# 8.5 Load and Process LLM BPI-IS Questionnaire Results
# ============================================================================
print('='*80)
print('Loading LLM BPI-IS Questionnaire Results')
print('='*80)

bpi_query = text("""
    SELECT 
        er.experiment_id,
        er.experiments_group_id,
        er.narrative_id,
        er.result_json
    FROM pain_narratives_app.evaluation_results er
    WHERE er.experiments_group_id IN (35, 36)
      AND er.result_type = 'BPI-IS'
""")

with Session(engine) as session:
    bpi_df = pd.DataFrame(session.exec(bpi_query).fetchall(),
                         columns=['experiment_id', 'experiments_group_id', 'narrative_id', 'result_json'])

print(f'✓ Loaded {len(bpi_df)} BPI-IS results')

# Extract BPI-IS scores and item responses
def extract_bpi_data(result_json):
    """Extract BPI-IS total score, subscales, and individual item scores"""
    if not result_json or not isinstance(result_json, dict):
        return {}
    
    bpi_total = result_json.get('total_score')
    data = {
        'bpi_total': bpi_total,
        # bpi_total_mean: Mean of all items (0-10 scale, matches real data format)
        'bpi_total_mean': bpi_total / 10.0 if bpi_total is not None else None,
    }
    
    # Subscales
    subscales = result_json.get('subscales', {})
    data['bpi_interference_avg'] = subscales.get('interference_avg')
    data['bpi_intensity_avg'] = subscales.get('intensity_avg')
    data['bpi_interference_total'] = subscales.get('interference_total')
    data['bpi_intensity_total'] = subscales.get('intensity_total')
    
    # Individual item responses (from raw_data.responses - it's a LIST of {code, value})
    raw_data = result_json.get('raw_data', {})
    responses = raw_data.get('responses', [])
    
    # Convert list to dict by code
    responses_dict = {}
    if isinstance(responses, list):
        for item in responses:
            if isinstance(item, dict):
                code = item.get('code', '')
                value = item.get('value')
                responses_dict[code] = value
    
    # BPI interference items (Q1: items 1-7) - note: Q1_4 might be missing
    for i in range(1, 8):
        key = f'BPI_Q1_{i}'
        data[f'bpiq1{i}'] = responses_dict.get(key)
    
    # BPI intensity items (Q2-Q5: items 8-11)
    data['bpiq28'] = responses_dict.get('BPI_Q2_8')
    data['bpiq39'] = responses_dict.get('BPI_Q3_9')
    data['bpiq410'] = responses_dict.get('BPI_Q4_10')
    data['bpiq511'] = responses_dict.get('BPI_Q5_11')
    
    # Model reasoning
    data['model_reasoning'] = raw_data.get('model_reasoning')
    
    # Persona info
    persona = raw_data.get('persona', {})
    data['persona_name'] = persona.get('name')
    data['persona_traits'] = persona.get('traits')
    
    return data

# Apply extraction
bpi_data = bpi_df['result_json'].apply(extract_bpi_data)
bpi_data_df = pd.DataFrame(bpi_data.tolist())

# Combine with experiment data
df_llm_bpi = pd.concat([
    bpi_df[['experiment_id', 'experiments_group_id', 'narrative_id']],
    bpi_data_df
], axis=1)

# Add participant_id and model
df_llm_bpi['participant_id'] = df_llm_bpi['experiment_id'].map(exp_to_participant)
df_llm_bpi['model'] = df_llm_bpi['experiment_id'].map(exp_to_model)
df_llm_bpi['run_number'] = df_llm_bpi['experiments_group_id'].map({35: 1, 36: 1})

# Filter to only matched records
df_llm_bpi = df_llm_bpi[df_llm_bpi['participant_id'].notna()].copy()

# Reorder columns
interference_cols = [f'bpiq1{i}' for i in range(1, 8)]
intensity_cols = ['bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'bpi_total', 'bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg', 
              'bpi_interference_total', 'bpi_intensity_total'] + interference_cols + intensity_cols + \
             ['persona_name', 'persona_traits', 'model_reasoning']
df_llm_bpi = df_llm_bpi[[c for c in cols_order if c in df_llm_bpi.columns]]

print(f'\n✓ Extracted BPI-IS data for {len(df_llm_bpi)} records')
print(f'\nBPI-IS Scores:')
print(f'  Total Mean (0-10 scale): {df_llm_bpi["bpi_total_mean"].mean():.2f}')
print(f'  Interference Mean: {df_llm_bpi["bpi_interference_avg"].mean():.2f}')
print(f'  Intensity Mean: {df_llm_bpi["bpi_intensity_avg"].mean():.2f}')

print(f'\nSample data:')
print(df_llm_bpi[['participant_id', 'bpi_total', 'bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg']].head())

In [ ]:
# ============================================================================
# 8.6 Load and Process LLM TSK-11SV Questionnaire Results
# ============================================================================
print('='*80)
print('Loading LLM TSK-11SV Questionnaire Results')
print('='*80)

tsk_query = text("""
    SELECT 
        er.experiment_id,
        er.experiments_group_id,
        er.narrative_id,
        er.result_json
    FROM pain_narratives_app.evaluation_results er
    WHERE er.experiments_group_id IN (35, 36)
      AND er.result_type = 'TSK-11SV'
""")

with Session(engine) as session:
    tsk_df = pd.DataFrame(session.exec(tsk_query).fetchall(),
                         columns=['experiment_id', 'experiments_group_id', 'narrative_id', 'result_json'])

print(f'✓ Loaded {len(tsk_df)} TSK-11SV results')

# Extract TSK-11SV scores and item responses
def extract_tsk_data(result_json):
    """Extract TSK-11SV total score and individual item scores"""
    if not result_json or not isinstance(result_json, dict):
        return {}
    
    data = {
        'tsk_total': result_json.get('total_score'),
    }
    
    # Individual item scores (from raw_data.responses)
    raw_data = result_json.get('raw_data', {})
    responses = raw_data.get('responses', [])
    
    # TSK responses are stored as array of {code: 'TSK_01', value: N}
    for resp in responses:
        code = resp.get('code', '')
        value = resp.get('value')
        if code.startswith('TSK_'):
            item_num = code.replace('TSK_', '')
            col_name = f'tsk_{item_num.zfill(2)}'
            data[col_name] = str(value) if value is not None else None
    
    # Model reasoning
    data['model_reasoning'] = raw_data.get('model_reasoning')
    
    # Persona info
    persona = raw_data.get('persona', {})
    data['persona_name'] = persona.get('name')
    data['persona_traits'] = persona.get('traits')
    
    return data

# Apply extraction
tsk_data = tsk_df['result_json'].apply(extract_tsk_data)
tsk_data_df = pd.DataFrame(tsk_data.tolist())

# Combine with experiment data
df_llm_tsk = pd.concat([
    tsk_df[['experiment_id', 'experiments_group_id', 'narrative_id']],
    tsk_data_df
], axis=1)

# Add participant_id and model
df_llm_tsk['participant_id'] = df_llm_tsk['experiment_id'].map(exp_to_participant)
df_llm_tsk['model'] = df_llm_tsk['experiment_id'].map(exp_to_model)
df_llm_tsk['run_number'] = df_llm_tsk['experiments_group_id'].map({35: 1, 36: 1})

# Filter to only matched records
df_llm_tsk = df_llm_tsk[df_llm_tsk['participant_id'].notna()].copy()

# Reorder columns
item_cols = [f'tsk_{str(i).zfill(2)}' for i in range(1, 12)]
cols_order = ['participant_id', 'experiment_id', 'experiments_group_id', 'run_number', 'model',
              'tsk_total'] + item_cols + ['persona_name', 'persona_traits', 'model_reasoning']
df_llm_tsk = df_llm_tsk[cols_order]

print(f'\n✓ Extracted TSK-11SV data for {len(df_llm_tsk)} records')
print(f'\nTSK-11SV Total Score:')
print(f'  Mean: {df_llm_tsk["tsk_total"].mean():.2f}')
print(f'  SD: {df_llm_tsk["tsk_total"].std():.2f}')
print(f'  Range: [{df_llm_tsk["tsk_total"].min()}, {df_llm_tsk["tsk_total"].max()}]')

print(f'\nSample data:')
print(df_llm_tsk[['participant_id', 'tsk_total'] + item_cols[:3]].head())

In [ ]:
# ============================================================================
# 8.7 Save LLM Synthetic Data to New Schema
# ============================================================================
print('='*80)
print('Saving LLM Synthetic Data to New Schema')
print('='*80)

# ============================================================================
# Table 1: LLM Dimension Evaluation
# ============================================================================
print('\n' + '-'*80)
print('Table: llm_dimension_evaluation')
print('-'*80)

df_llm_dimensions.to_sql(
    'llm_dimension_evaluation',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_llm_dimensions)} rows')
print(f'  Columns: {len(df_llm_dimensions.columns)}')

# ============================================================================
# Table 2: LLM PCS Results
# ============================================================================
print('\n' + '-'*80)
print('Table: llm_pcs_results')
print('-'*80)

df_llm_pcs.to_sql(
    'llm_pcs_results',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_llm_pcs)} rows')
print(f'  Columns: {len(df_llm_pcs.columns)}')

# ============================================================================
# Table 3: LLM BPI Results
# ============================================================================
print('\n' + '-'*80)
print('Table: llm_bpi_results')
print('-'*80)

df_llm_bpi.to_sql(
    'llm_bpi_results',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_llm_bpi)} rows')
print(f'  Columns: {len(df_llm_bpi.columns)}')

# ============================================================================
# Table 4: LLM TSK Results
# ============================================================================
print('\n' + '-'*80)
print('Table: llm_tsk_results')
print('-'*80)

df_llm_tsk.to_sql(
    'llm_tsk_results',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f'✓ Saved {len(df_llm_tsk)} rows')
print(f'  Columns: {len(df_llm_tsk.columns)}')

# ============================================================================
# Verify All LLM Tables
# ============================================================================
print('\n' + '='*80)
print('VERIFICATION')
print('='*80)

llm_tables = ['llm_dimension_evaluation', 'llm_pcs_results', 'llm_bpi_results', 'llm_tsk_results']

with Session(engine) as session:
    for table in llm_tables:
        count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table}')).first()
        null_count = session.exec(text(f'SELECT COUNT(*) FROM {NEW_SCHEMA}.{table} WHERE participant_id IS NULL')).first()
        print(f'✓ {table}: {count[0]} rows (NULL participant_ids: {null_count[0]})')

print('\n✓ All LLM synthetic data tables created successfully!')

## 9. System Usability Scale (SUS) Data Migration

This section migrates SUS usability survey responses from expert users to the ACM schema.

**SUS Overview:**
- 10-item questionnaire measuring perceived usability (1-5 Likert scale)
- Scoring: Odd items (q1,3,5,7,9) = response-1; Even items (q2,4,6,8,10) = 5-response
- Final score = sum × 2.5 (range 0-100)

**Benchmarks:**
- <50: F (Poor)
- 50-62: D (Marginal)  
- 62-68: C (Average)
- 68-80: B (Good)
- 80-90: A (Excellent)
- >90: A+ (Best Imaginable)

In [ ]:
# ============================================================================
# 9.1 Load SUS Data and Migrate to Database
# ============================================================================
print('='*80)
print('Loading SUS Usability Data')
print('='*80)

# Load SUS responses from CSV
sus_df = pd.read_csv('../data/sus_responses.csv')

print(f"\nTotal SUS responses: {len(sus_df)}")
print(f"Columns: {list(sus_df.columns)}")

# Display summary statistics
print(f"\n{'='*80}")
print("SUS Score Summary")
print('='*80)
print(f"Mean SUS Score: {sus_df['sus_score'].mean():.1f}")
print(f"Std SUS Score: {sus_df['sus_score'].std():.1f}")
print(f"Min SUS Score: {sus_df['sus_score'].min():.1f}")
print(f"Max SUS Score: {sus_df['sus_score'].max():.1f}")

# Grade assignment
mean_score = sus_df['sus_score'].mean()
if mean_score >= 90:
    grade = "A+ (Best Imaginable)"
elif mean_score >= 80:
    grade = "A (Excellent)"
elif mean_score >= 68:
    grade = "B (Good)"
elif mean_score >= 62:
    grade = "C (Average)"
elif mean_score >= 50:
    grade = "D (Marginal)"
else:
    grade = "F (Poor)"
    
print(f"Overall Grade: {grade}")

# Display respondent professions
print(f"\n{'='*80}")
print("Respondent Demographics")
print('='*80)
print(f"\nProfessions:")
print(sus_df['profession'].value_counts().to_string())

# Prepare DataFrame for database insertion (add sus_id column)
sus_db_df = sus_df.copy()
sus_db_df['sus_id'] = range(1, len(sus_db_df) + 1)
sus_db_df = sus_db_df[['sus_id', 'email', 'username', 'profession', 'age', 
                        'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10',
                        'sus_score', 'completed_date']]

print(f"\n{'='*80}")
print("Saving SUS Data to Database")
print('='*80)

# Save to database
sus_db_df.to_sql(
    'sus_usability_results',
    engine,
    schema=NEW_SCHEMA,
    if_exists='replace',
    index=False
)

print(f"✓ Saved {len(sus_db_df)} SUS records to {NEW_SCHEMA}.sus_usability_results")

# Verify
verify_query = text(f"SELECT COUNT(*) as count FROM {NEW_SCHEMA}.sus_usability_results")
with engine.connect() as conn:
    count = conn.execute(verify_query).scalar()
print(f"✓ Verified: {count} records in database")